<a href="https://colab.research.google.com/github/sebabecerra/Econometria-II-DEN/blob/main/Tarea1_EconometriaII.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econometría II — Tarea 1
**Mínimos Cuadrados Ordinarios, Predicción y Evaluación de Políticas**

Datos NLS. Modelo base: $lwage = \beta_0 + \beta_1\,educ + \beta_2\,exper + \beta_3\,exper^2 + u$, con $exper = age - educ - 6$.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

# Cargar datos
datos = pd.read_stata('NLS80V2.dta')   # ajustar la ruta si el archivo está en otra carpeta
datos[['lwage', 'educ', 'exper', 'age']].describe().round(3)

,lwage,educ,exper,age
count,935.000,935.000,935.000,935.000
mean,6.779,13.468,13.612,33.080
std,0.421,2.197,3.828,3.108
min,4.745,9.000,5.000,28.000
25%,6.506,12.000,11.000,30.000
50%,6.808,12.000,13.000,33.000
75%,7.056,16.000,17.000,36.000
max,8.032,18.000,23.000,38.000


## Pregunta 1 — Estimación del modelo por MCO

Se reportan coeficientes y errores estándar.

In [2]:
datos['exper2'] = datos['exper']**2
X = sm.add_constant(datos[['educ', 'exper', 'exper2']])
modelo = sm.OLS(datos['lwage'], X).fit()

modelo_hc1 = modelo.get_robustcov_results(cov_type='HC1')

tabla1 = pd.DataFrame({
    'coeficiente': modelo.params,
    'error estándar': modelo.bse,
    'error estándar HC1': modelo_hc1.bse
})

print(tabla1.round(5))
print(f"\nn = {int(modelo.nobs)},  R² = {modelo.rsquared:.4f}")

        coeficiente  error estándar  error estándar HC1
const       5.05292         0.21333             0.20796
educ        0.08383         0.00725             0.00740
exper       0.06710         0.02394             0.02354
exper2     -0.00158         0.00084             0.00082

n = 935,  R² = 0.1282


In [3]:
print(modelo.summary())
print(modelo_hc1.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.128
Model:                            OLS   Adj. R-squared:                  0.125
Method:                 Least Squares   F-statistic:                     45.64
Date:                Thu, 13 Aug 2026   Prob (F-statistic):           1.60e-27
Time:                        12:09:22   Log-Likelihood:                -453.49
No. Observations:                 935   AIC:                             915.0
Df Residuals:                     931   BIC:                             934.3
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.0529      0.213     23.686      0.0

Obteniendo HC1

$$
\widehat{\operatorname{Var}}_{\mathrm{HC1}}(\hat{\beta})
=
\frac{n}{n-k}
(X'X)^{-1}
\left[
X'(\hat{u}_i^2)X
\right]
(X'X)^{-1}
$$

In [4]:
u = modelo.resid
n, k = X.shape

XX_inv = np.linalg.inv(X.T @ X)

V_HC1 = (n / (n-k)) * XX_inv @ (X.T @ np.diag(u**2) @ X) @ XX_inv

se_HC1 = np.sqrt(np.diag(V_HC1))

print(pd.Series(se_HC1, index=X.columns).round(5))

const     0.20796
educ      0.00740
exper     0.02354
exper2    0.00082
dtype: float64


In [5]:
# Coeficientes que se usan en todo lo que sigue
b1 = modelo.params['educ']
b2 = modelo.params['exper']
b3 = modelo.params['exper2']

## Pregunta 2 — Efecto de aumentar en 1 año la educación de todos

La experiencia depende de la edad: $exper = age - educ - 6$. A edad fija, un año más de educación reduce la experiencia en un año, de modo que la política mueve las **tres** variables del modelo: $educ \to educ+1$, $exper \to exper-1$ y $exper^2 \to (exper-1)^2$.

**Efecto individual.** Diferencia con y sin política:

$$\Delta_i = \big[\beta_0 + \beta_1(educ_i+1) + \beta_2(exper_i-1) + \beta_3(exper_i-1)^2\big] - \big[\beta_0 + \beta_1\,educ_i + \beta_2\,exper_i + \beta_3\,exper_i^2\big]$$

Tenemos:

$$\Delta_i = \beta_1 - \beta_2 + \beta_3(1 - 2\,exper_i)$$

El efecto es **heterogéneo** en $exper_i$ depende de la experiencia de cada individuo.

El cambio promedio es:

$$\bar\Delta = \beta_1 - \beta_2 + \beta_3(1 - 2\bar e)$$

donde $\bar e$ es la experiencia promedio *observada* (pre-política)

Entonces $\hat{\bar\Delta} = 0.08383 - 0.06710 - 0.00158\,(1 - 2\cdot 13.6118) = 0.05822 \approx +5.8\%$ en el salario promedio

In [6]:
e_barra = datos['exper'].mean()
efecto2 = b1 - b2 + b3 * (1 - 2*e_barra)
print(f"exper promedio (observada, pre-política): {e_barra:.4f}")
print(f"Efecto sobre lwage promedio: {efecto2:.5f}  (~{100*efecto2:.2f}% en el salario)")

exper promedio (observada, pre-política): 13.6118
Efecto sobre lwage promedio: 0.05822  (~5.82% en el salario)


## Pregunta 3 — El efecto como coeficiente de una regresión (reparametrización)

Se puede usar la politica como restrccion de parametros reescribiendo el mismo modelo como una combinación lineal de coeficientes: $\theta \equiv \beta_1 - \beta_2 + \beta_3(1 - 2\bar e)$. . La ventaja: MCO reporta automáticamente el error estándar de sus coeficientes, así que $\theta$ sale con su inferencia directa de la tabla.

**Paso 1 — Despejar $\beta_1$ de la definición de $\theta$:**

$$\beta_1 = \theta + \beta_2 - \beta_3(1 - 2\bar e)$$

**Paso 2 — Sustituir en el modelo original:**

$$lwage = \beta_0 + \big[\theta + \beta_2 - \beta_3(1 - 2\bar e)\big]\,educ + \beta_2\,exper + \beta_3\,exper^2 + u$$

**Paso 3 — Reagrupar juntando lo que multiplica a cada coeficiente:**

$$lwage = \beta_0 + \theta\,educ + \beta_2{(educ + exper)} + \beta_3{\big(exper^2 - (1 - 2\bar e)\,educ\big)} + u$$

Las nuevas variables salen solas del reagrupamiento: $z_1 = educ + exper$ y $z_2 = exper^2 - (1-2\bar e)\,educ$. Al regresar $lwage$ sobre $(educ,\, z_1,\, z_2)$, el coeficiente de $educ$ **es** $\theta$.



**Resultado:** $\hat\theta = 0.05822$ ($= \hat{\bar\Delta}$ de la P2), con $SE(\hat\theta) = 0.00596$ y $t = 9.77$. El test asociado contrasta $H_0\!: \theta = 0$ — *"la política no tiene efecto sobre el salario promedio"* — que se rechaza contundentemente. Nótese que no es la hipótesis $\beta_1 = 0$: son preguntas distintas.

In [7]:
a = 1 - 2*e_barra
datos['z1'] = datos['educ'] + datos['exper']
datos['z2'] = datos['exper2'] - a * datos['educ']   # = exper2 + (2ē-1)·educ

X3 = sm.add_constant(datos[['educ', 'z1', 'z2']])
modelo3 = sm.OLS(datos['lwage'], X3).fit()

print(f"θ (coef. de educ):    {modelo3.params['educ']:.5f}   (= efecto P2: {efecto2:.5f})")
print(f"SE(θ):                {modelo3.bse['educ']:.5f}")
print(f"t = {modelo3.tvalues['educ']:.3f}   ->  H0: 'la política no tiene efecto'")

θ (coef. de educ):    0.05822   (= efecto P2: 0.05822)
SE(θ):                0.00596
t = 9.765   ->  H0: 'la política no tiene efecto'


In [8]:
print(modelo3.summary())

                            OLS Regression Results                            
Dep. Variable:                  lwage   R-squared:                       0.128
Model:                            OLS   Adj. R-squared:                  0.125
Method:                 Least Squares   F-statistic:                     45.64
Date:                Thu, 13 Aug 2026   Prob (F-statistic):           1.60e-27
Time:                        12:09:34   Log-Likelihood:                -453.49
No. Observations:                 935   AIC:                             915.0
Df Residuals:                     931   BIC:                             934.3
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.0529      0.213     23.686      0.0

In [12]:
print("z1 recupera β2:", round(modelo3.params['z1'], 5), "=", round(b2, 5))
print("z2 recupera β3:", round(modelo3.params['z2'], 5), "=", round(b3, 5))
print("Mismo R²:", round(modelo.rsquared, 6), "=", round(modelo3.rsquared, 6))



z1 recupera β2: 0.0671 = 0.0671
z2 recupera β3: -0.00158 = -0.00158
Mismo R²: 0.128218 = 0.128218


## Pregunta 4 — Política: educación mínima de 12 años

Aumento individual: $d_i = \max(12 - educ_i,\, 0)$ (heterogéneo; $d_i = 0$ para los no afectados). Por la mecánica de la experiencia potencial: $educ_i \to educ_i + d_i$, $exper_i \to exper_i - d_i$. El efecto individual:

$$\Delta_i = \beta_1 d_i - \beta_2 d_i + \beta_3\,(d_i^2 - 2\,exper_i\, d_i)$$

Con $d_i=1$ se recupera la fórmula de la P2; con $d_i=0$, $\Delta_i=0$. El promedio se toma sobre **toda** la muestra (los ceros diluyen). No hay atajo de medias: $d_i$ y $exper_i$ aparecen multiplicados, así que se promedia la columna de efectos individuales.

In [13]:
d = (12 - datos['educ']).clip(lower=0)

delta_i = b1*d - b2*d + b3*(d**2 - 2*datos['exper']*d)
efecto4 = delta_i.mean()

print(f"Personas afectadas: {(d > 0).sum()} de {len(datos)}")
print(f"Aumento promedio de educación (d̄): {d.mean():.4f} años")
print(f"Efecto sobre lwage promedio: {efecto4:.5f}  (~{100*efecto4:.2f}% en el salario)")

Personas afectadas: 88 de 935
Aumento promedio de educación (d̄): 0.1529 años
Efecto sobre lwage promedio: 0.01083  (~1.08% en el salario)


In [14]:
# Verificación por predicción
educ_pol  = datos['educ'].clip(lower=12)
exper_pol = datos['exper'] - d
X_pol4 = sm.add_constant(pd.DataFrame({
    'educ': educ_pol, 'exper': exper_pol, 'exper2': exper_pol**2
}))
verif4 = (modelo.predict(X_pol4) - modelo.predict(X)).mean()
print(f"Verificación por predicción: {verif4:.5f}   (debe coincidir con {efecto4:.5f})")

Verificación por predicción: 0.01083   (debe coincidir con 0.01083)


*Nota (nivel vs. logaritmo):* el enunciado dice "nivel promedio de ingresos". El efecto reportado está en unidades de $lwage$ (cambio proporcional aproximado, $\times 100 \approx \%$). La traducción a niveles monetarios ($wage_i \cdot e^{\Delta_i}$, con residuo fijo) se muestra como complemento; su inferencia sería no lineal en $\hat\beta$ y queda fuera de la P5.

In [15]:
# Complemento: traducción a niveles (supuesto: residuo individual invariante a la política)
if 'wage' in datos.columns:
    wage_obs = datos['wage']
else:
    wage_obs = np.exp(datos['lwage'])
efecto4_niveles = (wage_obs * np.exp(delta_i) - wage_obs).mean()
print(f"Aumento promedio del salario en niveles: {efecto4_niveles:.2f} (unidades de wage)")

Aumento promedio del salario en niveles: 8.98 (unidades de wage)


## Pregunta 5 — Error estándar del efecto de la P4

El efecto es una **combinación lineal** de los coeficientes: agrupando el promedio por coeficiente,

$$\bar\Delta = \beta_1 w_1 + \beta_2 w_2 + \beta_3 w_3 = c'\beta, \qquad c = (0,\; \bar d,\; -\bar d,\; \overline{d^2 - 2\,exper\,d})'$$

Los pesos se tratan como fijos (análisis condicional en $X$); la única aleatoriedad es $\hat\beta$. La varianza de una combinación lineal requiere varianzas **y covarianzas**:

$$Var(c'\hat\beta) = c'\,\hat V\,c \;=\; \sum_j\sum_k c_j c_k \hat V_{jk} \qquad\Rightarrow\qquad SE = \sqrt{c'\hat V c}$$

In [16]:
# Vector de pesos c (orden: const, educ, exper, exper2)
w1 = d.mean()
w2 = -d.mean()
w3 = (d**2 - 2*datos['exper']*d).mean()
c  = np.array([0, w1, w2, w3])

print("c =", c.round(4))
print("Chequeo c'β̂ = efecto P4:", (c @ modelo.params.values).round(5), "=", efecto4.round(5))

c = [ 0.      0.1529 -0.1529 -5.2267]
Chequeo c'β̂ = efecto P4: 0.01083 = 0.01083


### 5a. Errores estándar clásicos ($\hat V = \hat\sigma^2 (X'X)^{-1}$)

In [17]:
V_clasica = modelo.cov_params().values
se_clasico = np.sqrt(c @ V_clasica @ c)
gl = int(modelo.df_resid)
from scipy import stats
tcrit = stats.t.ppf(0.975, gl)

print(f"SE clásico:  {se_clasico:.5f}")
print(f"t = {efecto4/se_clasico:.3f}  (gl = {gl})")
print(f"IC 95%: [{efecto4 - tcrit*se_clasico:.5f}, {efecto4 + tcrit*se_clasico:.5f}]")

# Verificación con t_test (misma cuenta hecha por statsmodels)
print("\n", modelo.t_test(c))

SE clásico:  0.00124
t = 8.741  (gl = 931)
IC 95%: [0.00840, 0.01326]

                              Test for Constraints                             
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.0108      0.001      8.741      0.000       0.008       0.013


### 5b. Construcción manual de $\hat V$ clásica (desde los residuos)

$\hat\sigma^2 = \dfrac{\sum_i \hat u_i^2}{n-k}$ con $\hat u_i = y_i - \hat y_i$; luego $\hat V = \hat\sigma^2 (X'X)^{-1}$.

In [18]:
u  = modelo.resid.values
Xm = X.values
n, k = Xm.shape

sigma2   = (u**2).sum() / (n - k)
V_manual = sigma2 * np.linalg.inv(Xm.T @ Xm)

print(f"σ̂² = {sigma2:.5f}")
print("V manual == cov_params:", np.allclose(V_manual, V_clasica))
print(f"SE manual del efecto: {np.sqrt(c @ V_manual @ c):.5f}")

σ̂² = 0.15512
V manual == cov_params: True
SE manual del efecto: 0.00124


### 5c. Errores estándar robustos HC1 (sándwich de White)

$$\hat V_{HC1} = \tfrac{n}{n-k}\,(X'X)^{-1}\Big(\textstyle\sum_i \hat u_i^2\, x_i x_i'\Big)(X'X)^{-1}$$

Los $\hat\beta$ (y el efecto puntual) **no cambian**; solo cambia la matriz que entra a $c'\hat V c$. Bajo homocedasticidad la "carne" colapsa: $\sum \hat u_i^2 x_i x_i' \approx \hat\sigma^2 X'X$ y el sándwich se reduce a la clásica.

In [19]:
# Construcción manual del sándwich
carne  = Xm.T @ np.diag(u**2) @ Xm          # Σ û_i² · x_i x_i'
pan    = np.linalg.inv(Xm.T @ Xm)
V_hc1m = (n/(n-k)) * pan @ carne @ pan

# Verificación contra statsmodels
modelo_r = sm.OLS(datos['lwage'], X).fit(cov_type='HC1')
print("V_HC1 manual == statsmodels:", np.allclose(V_hc1m, modelo_r.cov_params().values))

se_hc1 = np.sqrt(c @ V_hc1m @ c)
print(f"\nEfecto (idéntico):  {(c @ modelo_r.params.values):.5f}")
print(f"SE clásico:  {se_clasico:.5f}")
print(f"SE HC1:      {se_hc1:.5f}")
print(f"IC 95% (HC1): [{efecto4 - tcrit*se_hc1:.5f}, {efecto4 + tcrit*se_hc1:.5f}]")

V_HC1 manual == statsmodels: True

Efecto (idéntico):  0.01083
SE clásico:  0.00124
SE HC1:      0.00116
IC 95% (HC1): [0.00856, 0.01310]


### 5d. Verificación por reparametrización (misma receta de la P3)

Despejando $\beta_1$ de $\theta_4 = c'\beta$ y reagrupando, la regresión de $lwage$ sobre $\big(educ/\bar d,\; educ+exper,\; exper^2 - (w_3/\bar d)\,educ\big)$ entrega $\theta_4$ como coeficiente, con su SE directo en la tabla. Debe coincidir al decimal con 5a (y con 5c si se estima con `HC1`).

In [20]:
datos['x_star'] = datos['educ'] / d.mean()
datos['z2p']    = datos['exper2'] - (w3 / d.mean()) * datos['educ']

X5 = sm.add_constant(datos[['x_star', 'z1', 'z2p']])

m5_cl = sm.OLS(datos['lwage'], X5).fit()
m5_r  = sm.OLS(datos['lwage'], X5).fit(cov_type='HC1')

print(f"θ4 (coef x*):      {m5_cl.params['x_star']:.5f}   (= {efecto4:.5f})")
print(f"SE clásico:        {m5_cl.bse['x_star']:.5f}   (= {se_clasico:.5f})")
print(f"SE HC1:            {m5_r.bse['x_star']:.5f}   (= {se_hc1:.5f})")
print(f"Mismo R² que el modelo original: {np.isclose(m5_cl.rsquared, modelo.rsquared)}")

θ4 (coef x*):      0.01083   (= 0.01083)
SE clásico:        0.00124   (= 0.00124)
SE HC1:            0.00116   (= 0.00116)
Mismo R² que el modelo original: True


## Resumen de resultados

| Pregunta | Objeto | Resultado |
|---|---|---|
| 1 | $\hat\beta$, SE | tabla MCO |
| 2 | $\bar\Delta_{+1} = \beta_1-\beta_2+\beta_3(1-2\bar e)$ | `efecto2` |
| 3 | $\theta$ como coef. de regresión reparametrizada | `modelo3` |
| 4 | $\bar\Delta_{12} = c'\hat\beta$ | `efecto4` |
| 5 | $SE = \sqrt{c'\hat V c}$ (clásico y HC1, 3 vías coincidentes) | `se_clasico`, `se_hc1` |

**Interpretación P4–P5:** la política de llevar a 12 años la educación mínima aumenta el salario promedio en $\approx 100\cdot\bar\Delta\,\%$; con $t = \bar\Delta / SE$ se contrasta $H_0$: *la política no tiene efecto sobre el salario promedio* (que no es lo mismo que $\beta_1 = 0$).